# 01 · Anomaly Detection Taxonomy & Statistical Baselines

## What this notebook covers
Anomaly detection is the task of finding observations that deviate from expected behaviour without having labelled examples of anomalies. This notebook maps the full taxonomy and implements the statistical baselines that serve as reference points for every more complex method.

## The three anomaly types
| Type | Definition | Example |
|------|-----------|---------|
| **Point** | A single observation is anomalous in isolation | A single transaction of $50,000 on a $500/month card |
| **Contextual** | Anomalous given context but not in isolation | 30°C in February (normal in July) |
| **Collective** | A sequence of observations is anomalous together | Normal individual heartbeats forming an abnormal rhythm |

## Why the baselines still matter
Z-score, IQR, and Grubbs' test are fast, interpretable, and require no training. For univariate data with known distributions they often match or beat complex models. They are the benchmark every new method must beat.

**Grubbs' test** (1950) is the classical parametric test for a single outlier in a normally-distributed univariate dataset. It assumes normality and is designed for small samples. For modern multivariate or non-normal data, use Isolation Forest or LOF (NB 02). We implement it here for completeness and as a sanity-check baseline.

**One-class SVM** is covered narratively: it works by finding the minimum-volume hypersphere in kernel space enclosing the training data. It performs well on low-dimensional, well-separated data but scales as O(n²) in training and is highly sensitive to the `nu` and `gamma` hyperparameters. Isolation Forest and LOF dominate it in practice for tabular data; PatchCore (NB 09) dominates it for images.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# Synthetic univariate dataset with a few injected outliers
n_normal = 200
data_normal = np.random.normal(loc=50, scale=5, size=n_normal)
outliers = np.array([85, 90, 10, 8])
data = np.concatenate([data_normal, outliers])
true_labels = np.array([0] * n_normal + [1] * len(outliers))  # 1 = anomaly

print(f"Dataset: {len(data)} points, {true_labels.sum()} true anomalies")
print(f"Mean: {data.mean():.2f}  Std: {data.std():.2f}  Min: {data.min():.2f}  Max: {data.max():.2f}")

In [ ]:
# ── Z-score detector ───────────────────────────────────────────────────────────
def zscore_detector(x, threshold=3.0):
    z = np.abs(stats.zscore(x))
    return z > threshold, z

flags_z, z_scores = zscore_detector(data)
print(f"Z-score (threshold=3): {flags_z.sum()} flagged")
print(f"  Precision: {(flags_z & true_labels.astype(bool)).sum() / flags_z.sum():.2f}")
print(f"  Recall   : {(flags_z & true_labels.astype(bool)).sum() / true_labels.sum():.2f}")

In [ ]:
# ── IQR detector ──────────────────────────────────────────────────────────────
def iqr_detector(x, k=1.5):
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    flags = (x < q1 - k * iqr) | (x > q3 + k * iqr)
    return flags

flags_iqr = iqr_detector(data)
print(f"IQR (k=1.5): {flags_iqr.sum()} flagged")
print(f"  Precision: {(flags_iqr & true_labels.astype(bool)).sum() / flags_iqr.sum():.2f}")
print(f"  Recall   : {(flags_iqr & true_labels.astype(bool)).sum() / true_labels.sum():.2f}")

In [ ]:
# ── Grubbs' test ──────────────────────────────────────────────────────────────
def grubbs_test(x, alpha=0.05):
    """One-sided Grubbs test: detects the single most extreme outlier."""
    n = len(x)
    mean, std = x.mean(), x.std(ddof=1)
    G = np.abs(x - mean).max() / std
    # Critical value via t-distribution
    t_crit = stats.t.ppf(1 - alpha / (2 * n), df=n - 2)
    G_crit = ((n - 1) / np.sqrt(n)) * np.sqrt(t_crit**2 / (n - 2 + t_crit**2))
    outlier_idx = np.argmax(np.abs(x - mean))
    return G, G_crit, G > G_crit, outlier_idx

G, G_crit, is_outlier, outlier_idx = grubbs_test(data)
print(f"Grubbs test: G={G:.3f}  G_crit={G_crit:.3f}  outlier_detected={is_outlier}")
print(f"  Most extreme point: {data[outlier_idx]:.2f} at index {outlier_idx}")

In [ ]:
# Statistical baseline pipeline for multivariate data
def statistical_anomaly_pipeline(X, zscore_thresh=3.0, iqr_k=1.5):
    """Per-feature z-score and IQR flags; combined OR rule."""
    n, p = X.shape
    z_flags   = np.zeros(n, dtype=bool)
    iqr_flags = np.zeros(n, dtype=bool)
    for j in range(p):
        z_flags   |= (np.abs(stats.zscore(X[:, j])) > zscore_thresh)
        iqr_flags |= iqr_detector(X[:, j], iqr_k)
    return z_flags, iqr_flags

# Demo on 2D dataset
X2d, _ = make_blobs(n_samples=300, centers=1, cluster_std=1.0, random_state=0)
X2d = np.vstack([X2d, [[8, 8], [-7, -7], [9, -8]]])  # inject outliers
z2d, iqr2d = statistical_anomaly_pipeline(X2d)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, flags, title in zip(axes, [z2d, iqr2d], ["Z-score flags", "IQR flags"]):
    ax.scatter(X2d[~flags, 0], X2d[~flags, 1], c="steelblue", s=15, alpha=0.6, label="Normal")
    ax.scatter(X2d[flags,  0], X2d[flags,  1], c="red", s=60, marker="x", label="Flagged", zorder=5)
    ax.set_title(title); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{base}/01_baselines.png", dpi=120)
plt.show()
print(f"Z-score flagged: {z2d.sum()}  IQR flagged: {iqr2d.sum()}")